<a href="https://www.kaggle.com/code/shravanpadhar/suzuki-stock-v94-2-accuracy?scriptVersionId=312107277" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Suzuki Stock Analytics: Real-Time Prototype
System Status:  Active | Environment: localhost:8501

This notebook serves as the backend engine for the Suzuki Motor Corporation (7269.T) stock tracking dashboard. It processes live market data via the yFinance API and generates interactive visualizations for technical analysis.

[**Launch Dashboard**](http://localhost:8501)

In [ ]:
from IPython.display import Video, display

video_path = "/kaggle/input/recording-2026-04-16-221001/Recording 2026-04-16 221001.mp4"

# Adding mimetype='video/mp4' helps the browser recognize it
display(Video(video_path, width=900, embed=True, mimetype='video/mp4', html_attributes="controls autoplay muted loop"))

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load the specific CSV file from the Kaggle dataset
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "nilesh2042/suzuki-stock-data",
    "suzuki_stock_data.csv" # Specify the exact file name here
)
df.head()

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

In [ ]:
df = df[df['Volume'] > 0]

In [ ]:
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()

In [ ]:
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

In [ ]:
df['BB_Middle'] = df['Close'].rolling(window=20).mean()
std_dev = df['Close'].rolling(window=20).std()
df['BB_Upper'] = df['BB_Middle'] + (2 * std_dev)
df['BB_Lower'] = df['BB_Middle'] - (2 * std_dev)

In [ ]:

df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
df['Volatility_30'] = df['Log_Return'].rolling(window=30).std() * np.sqrt(252)
df.dropna(inplace=True)

In [ ]:
pip install mplfinance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import mplfinance as mpf

In [ ]:
# 1. Price Action: Candlestick Chart (Last 2 years)
df_2yr = df.last('2Y')
mpf.plot(df_2yr, type='candle', style='charles', title='Suzuki OHLC (Last 2 Years)', volume=True)

In [ ]:
# 2. Volume vs Price Analysis
plt.figure(figsize=(10, 6))
plt.scatter(df['Volume'], df['Close'], alpha=0.3, color='blue')
plt.title('Volume vs. Closing Price')
plt.xlabel('Volume')
plt.ylabel('Close Price (JPY)')
plt.show()

In [ ]:
# 3. Seasonality: Monthly Average Returns Heatmap
df_monthly = df.copy()
df_monthly['Month'] = df_monthly.index.month
monthly_returns = df_monthly.groupby('Month')['Log_Return'].mean().to_frame()

plt.figure(figsize=(8, 6))
sns.heatmap(monthly_returns, annot=True, cmap='RdYlGn', fmt=".4f")
plt.title('Monthly Average Returns Seasonality')
plt.show()

In [ ]:
# 4. Correlation Heatmap
plt.figure(figsize=(8, 6))
corr_matrix = df[['Open', 'High', 'Low', 'Close', 'Volume']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.show()

**Financial Insights:**
* **Seasonality:** August (+0.29%), May (+0.22%), and January (+0.17%) exhibit the highest historical average log returns. March (-0.28%) and February (-0.20%) exhibit the lowest.
* **Correlation:** Open, High, Low, and Close prices are nearly perfectly correlated (>0.99). Trading volume has a weak negative correlation (-0.17) with price movements.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

In [ ]:
adf_result = adfuller(df['Close'])
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")

In [ ]:
train_size = int(len(df) * 0.8)
train, test = df.iloc[:train_size], df.iloc[train_size:]

In [ ]:
prophet_train = train.reset_index()[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})
prophet_test = test.reset_index()[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})

In [ ]:
model = Prophet(daily_seasonality=False)
model.fit(prophet_train)

future = model.make_future_dataframe(periods=len(test))
forecast = model.predict(future)
predictions = forecast.iloc[-len(test):]['yhat'].values

rmse = np.sqrt(mean_squared_error(test['Close'], predictions))
mape = mean_absolute_percentage_error(test['Close'], predictions)
print(f"RMSE: {rmse:.4f} | MAPE: {mape:.4f}")

In [ ]:
df['Signal'] = np.where(df['SMA_20'] > df['SMA_50'], 1, 0)
df['Strategy_Return'] = df['Signal'].shift(1) * df['Log_Return']

cum_bh_return = np.exp(df['Log_Return'].cumsum())[-1]
cum_strat_return = np.exp(df['Strategy_Return'].cumsum())[-1]
print(f"Buy & Hold Cumulative Return: {cum_bh_return:.4f}x")
print(f"Golden Cross Cumulative Return: {cum_strat_return:.4f}x")

**Financial Insights & Model Interpretations:**
* **Stationarity:** The ADF statistic is -1.5193 with a p-value of 0.5239. The closing price time-series is non-stationary.
* **Model Accuracy:** Baseline modeling on an 80/20 split produces an RMSE of 366.01 JPY and a MAPE of 13.73%, meaning predictions deviate from actual closing prices by roughly 13.7%.
* **Backtesting:** The Golden Cross strategy yields a cumulative return multiplier of 1.02x, significantly underperforming the Buy & Hold strategy's 2.41x multiplier.